# API JSON Generator - Refactored

This notebook demonstrates a refactored product generator with Pydantic validation, OpenAI integration, and robust error handling.

## 1. Import Required Libraries

In [1]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Optional, List
from enum import Enum
import base64
import json
import os
import time
from io import BytesIO
from datetime import datetime

from openai import OpenAI
from PIL import Image
import requests

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. Setup OpenAI Client

In [2]:
api_key = os.environ.get("OPENAI_API_KEY")

if api_key:
    client = OpenAI(api_key=api_key)
    print("✓ OpenAI client initialized successfully")
else:
    client = None
    print("⚠ Warning: OPENAI_API_KEY not set. Validation demo will work, but OpenAI calls will fail.")

✓ OpenAI client initialized successfully


## 3. Define Pydantic Models

In [3]:
class ProductCategory(str, Enum):
    APPAREL = "Apparel"
    ACCESSORIES = "Accessories"
    FOOTWEAR = "Footwear"
    ELECTRONICS = "Electronics"
    HOME = "Home"
    BEAUTY = "Beauty"
    SPORTS = "Sports"
    OTHER = "Other"


class ProductGender(str, Enum):
    MEN = "Men"
    WOMEN = "Women"
    UNISEX = "Unisex"
    KIDS = "Kids"


class ProductInput(BaseModel):
    product_id: str = Field(..., min_length=1, max_length=50)
    product_name: str = Field(..., min_length=3, max_length=200)
    price: float = Field(..., gt=0, le=100000)
    currency: str = Field(default="USD", pattern=r"^[A-Z]{3}$")
    category: ProductCategory
    gender: Optional[ProductGender] = None
    color: Optional[str] = Field(default=None, max_length=50)
    brand: Optional[str] = Field(default=None, max_length=100)
    image_url: Optional[str] = None
    image_base64: Optional[str] = None
    tags: Optional[List[str]] = None

    @field_validator("product_name")
    @classmethod
    def clean_product_name(cls, value: str) -> str:
        return " ".join(value.strip().split())

    @field_validator("color")
    @classmethod
    def normalize_color(cls, value: Optional[str]) -> Optional[str]:
        if value is None:
            return None
        return value.strip().title()

    @field_validator("tags")
    @classmethod
    def clean_tags(cls, value: Optional[List[str]]) -> Optional[List[str]]:
        if value is None:
            return None
        cleaned = []
        seen = set()
        for tag in value:
            tag_clean = tag.strip().lower()
            if tag_clean and tag_clean not in seen:
                seen.add(tag_clean)
                cleaned.append(tag_clean)
        return cleaned

    @model_validator(mode="after")
    def check_image_provided(self):
        if not self.image_url and not self.image_base64:
            raise ValueError("Either image_url or image_base64 must be provided")
        return self


class GeneratedListing(BaseModel):
    title: str
    description: str
    features: List[str] = []
    tags: List[str] = []


class APIResponse(BaseModel):
    success: bool
    product_id: str
    generated_listing: Optional[GeneratedListing] = None
    error_message: Optional[str] = None
    processing_time_seconds: float
    timestamp: str = Field(default_factory=lambda: datetime.utcnow().isoformat())

print("✓ All Pydantic models defined")

✓ All Pydantic models defined


## 4. Validation Functions

In [4]:
def validate_product(json_data: dict):
    """
    Validate incoming JSON product data.
    Returns: (validated_product, errors)
    """
    try:
        product = ProductInput(**json_data)
        return product, None
    except Exception as e:
        errors = []
        if hasattr(e, "errors"):
            for error in e.errors():
                field = ".".join(str(loc) for loc in error["loc"])
                message = error["msg"]
                errors.append(f"{field}: {message}")
        else:
            errors.append(str(e))
        return None, errors


def display_validation_result(product_data: dict, test_name: str):
    print(f"\n--- {test_name} ---")
    product, errors = validate_product(product_data)

    if product:
        print("✓ VALIDATION PASSED")
        print(f"  Product ID: {product.product_id}")
        print(f"  Product Name: {product.product_name}")
        print(f"  Price: {product.price} {product.currency}")
        print(f"  Category: {product.category.value}")
        if product.gender:
            print(f"  Gender: {product.gender.value}")
        if product.color:
            print(f"  Color: {product.color}")
        if product.brand:
            print(f"  Brand: {product.brand}")
        if product.tags:
            print(f"  Tags: {product.tags}")
    else:
        print("✗ VALIDATION FAILED")
        for error in errors:
            print(f"  Error: {error}")

print("✓ Validation functions defined")

✓ Validation functions defined


## 5. Run Validation Demonstration

In [1]:
print("=" * 60)
print("PYDANTIC VALIDATION DEMONSTRATION")
print("=" * 60)

test_cases = [
    (
        "Test 1: Valid Product",
        {
            "product_id": "PROD-001",
            "product_name": "  Premium Cotton T-Shirt  ",
            "price": 29.99,
            "currency": "USD",
            "category": "Apparel",
            "gender": "Men",
            "color": "navy blue",
            "brand": "StyleCo",
            "image_url": "https://example.com/tshirt.jpg",
            "tags": ["casual", "CASUAL", "cotton", "comfort"]
        }
    ),
    (
        "Test 2: Negative Price",
        {
            "product_id": "PROD-002",
            "product_name": "Test Product",
            "price": -10.00,
            "category": "Apparel",
            "image_url": "https://example.com/image.jpg"
        }
    ),
    (
        "Test 3: Missing Product Name",
        {
            "product_id": "PROD-003",
            "price": 19.99,
            "category": "Apparel",
            "image_url": "https://example.com/image.jpg"
        }
    ),
    (
        "Test 4: Invalid Category",
        {
            "product_id": "PROD-004",
            "product_name": "Test Product",
            "price": 29.99,
            "category": "InvalidCategory",
            "image_url": "https://example.com/image.jpg"
        }
    ),
    (
        "Test 5: No Image Provided",
        {
            "product_id": "PROD-005",
            "product_name": "Product Without Image",
            "price": 49.99,
            "category": "Electronics"
        }
    ),
    (
        "Test 6: Invalid Currency Code",
        {
            "product_id": "PROD-006",
            "product_name": "Test Product",
            "price": 29.99,
            "currency": "dollars",
            "category": "Apparel",
            "image_url": "https://example.com/image.jpg"
        }
    ),
    (
        "Test 7: Product Name Too Short",
        {
            "product_id": "PROD-007",
            "product_name": "AB",
            "price": 29.99,
            "category": "Apparel",
            "image_url": "https://example.com/image.jpg"
        }
    ),
    (
        "Test 8: Price Too High",
        {
            "product_id": "PROD-008",
            "product_name": "Expensive Product",
            "price": 999999.99,
            "category": "Electronics",
            "image_url": "https://example.com/image.jpg"
        }
    )
]

for test_name, data in test_cases:
    display_validation_result(data, test_name)

print("\n" + "=" * 60)
print("VALIDATION DEMONSTRATION COMPLETE")
print("=" * 60)

PYDANTIC VALIDATION DEMONSTRATION


NameError: name 'display_validation_result' is not defined